# F11 · TwinCheck demo with the image card (Feba)

**Runtime → T4 GPU**, switch on **Notebook access** for `GITHUB_TOKEN` and `HF_TOKEN` in the 🔑 panel, then run the cells in order. No Google Drive needed: the detector (`models/F5/vit_b16_s42.pt`, Git LFS), the 96 px cache and the F3 JPEG pool (`models/F11/jpeg_pool.json`) are in the repo.

1. Setup: repo, checkpoint (`git lfs pull`, with a direct download as fallback; it must print `checkpoint: 343 MB`), packages. The text side uses the trained verdict head in fast mode.
2. Tests.
3. Preprocessing check: 10 cached test images scored again and compared with `results/F10/preds_vit_b16_s42_check.csv` (must differ by < 1e-3); pushes `results/F11/preprocess_check.csv` (ends with `PUSHED`).
4. The six text + image cases scored by the detector: known label, P(AI), correct or not.
5. Launch the demo (public `gradio.live` link). Use the **Text + image cases** buttons (1 real photo + true claim, 2 AI image + true caption, 3 AI image + false claim, 4 AI image the detector misses, 5a/5b image twin) or paste a message and upload your own picture. You get the claim card, the image card (AI-generated / Real with P(AI); known label and ✓/✗ for the cases), and a final result for text and image together. The detector is reliable on BigGAN-style images and misses most images from newer generators, so uploads from today's AI tools are often called Real. The cell keeps running while the demo is open.


In [ ]:
from google.colab import userdata
import os
REPO = '/content/twincheck'
BRANCH = 'feba/F11-demo-vision'
if not os.path.exists(REPO):
    !git clone -q https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
CKPT = 'models/F5/vit_b16_s42.pt'
!which git-lfs || apt-get -qq install -y git-lfs
!git lfs install
!git lfs pull --include {CKPT}
if os.path.getsize(CKPT) < 1_000_000:
    !wget -q -O {CKPT} https://media.githubusercontent.com/media/himaallu/Group-Project/{BRANCH}/{CKPT}
size_mb = os.path.getsize(CKPT) / 1e6
print(f'checkpoint: {size_mb:.0f} MB')
assert size_mb > 300, 'The detector checkpoint did not download: rerun this cell'
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
!pip install -q -U gradio rank_bm25 faiss-cpu
!python -m spacy download -q en_core_web_sm
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
os.environ.update(TWINCHECK_NLI='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7',
                  TWINCHECK_VERDICT='models/verdict_twin_s42', TWINCHECK_EXTRACTOR='split',
                  TWINCHECK_EXPLAINER='template', TWINCHECK_SPACY='en_core_web_sm')


In [ ]:
!python -m pytest -q tests/demo

In [ ]:
!python -m src.vision.infer --verify-cache data/processed/vision/genimage_debiased96_jpeg.pkl --n 10 --out results/F11/preprocess_check.csv
!git add results/F11/preprocess_check.csv && git commit -q -m "[F11] Preprocessing check: 10 cached test images vs F10" && git push -q origin {BRANCH} && echo PUSHED


In [ ]:
from src.demo.image_examples import load_examples
from src.vision.infer import check_image, load_detector, load_jpeg_pool
model, pool = load_detector('models/F5/vit_b16_s42.pt'), load_jpeg_pool('models/F11/jpeg_pool.json')
for ex in load_examples():
    r = check_image(ex['image'], model, pool, blob=ex['blob'])
    right = (r['label'] == 'AI-generated') == ex['truth'].startswith('AI')
    print(f"{ex['title']:34s} known: {ex['truth']:38s} detector: {r['label']:13s} P(AI)={r['p_ai']:.3f}  {'correct' if right else 'WRONG'}")
del model


In [ ]:
!python -m src.demo.app --share